# PSCon Product Table Preparation

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "../..")
from Shared.product_embedding import create_product_embeddings

## 1. Product Facts Table

In [2]:
lines = pd.Series(
    Path("./Raw Dataset/dataset/knowledgeGraph_en.txt").read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()
)

facts = lines.str.split(", ", n=2, expand=True)
facts.columns = ["product_id", "key", "value"]

facts = facts.replace(r"^\s*$", pd.NA, regex=True).dropna().drop_duplicates()

valid_product_ids = set(
    facts.loc[facts["key"].str.casefold().eq("title"), "product_id"]
)
facts = facts.loc[facts["product_id"].isin(valid_product_ids)].copy()


## 2. Product Table

In [3]:
products = []

for product_id, rows in facts.groupby("product_id", sort=False):
    title_mask = rows["key"].str.casefold().eq("title")
    product_name = rows.loc[title_mask, "value"].iloc[0]
    details = [
        f"{row.key}: {row.value}"
        for row in rows.loc[~title_mask].itertuples(index=False)
    ]

    products.append({
        "product_id": product_id,
        "product_name": product_name,
        "search_text": " | ".join([
            f"Title: {product_name}",
            *details,
        ]),
    })

product_table = pd.DataFrame(products)
product_table.insert(0, "product_index", range(len(product_table)))


## 3. Add Embeddings

In [4]:
embedding_input = product_table.rename(
    columns={"product_name": "title"}
)
vectors = create_product_embeddings(embedding_input)
product_table["embedding"] = list(vectors)

output_table = product_table.copy()
output_table["embedding"] = [vector.tolist() for vector in vectors]

Embedded 100/22,202
Embedded 200/22,202
Embedded 300/22,202
Embedded 400/22,202
Embedded 500/22,202
Embedded 600/22,202
Embedded 700/22,202
Embedded 800/22,202
Embedded 900/22,202
Embedded 1,000/22,202
Embedded 1,100/22,202
Embedded 1,200/22,202
Embedded 1,300/22,202
Embedded 1,400/22,202
Embedded 1,500/22,202
Embedded 1,600/22,202
Embedded 1,700/22,202
Embedded 1,800/22,202
Embedded 1,900/22,202
Embedded 2,000/22,202
Embedded 2,100/22,202
Embedded 2,200/22,202
Embedded 2,300/22,202
Embedded 2,400/22,202
Embedded 2,500/22,202
Embedded 2,600/22,202
Embedded 2,700/22,202
Embedded 2,800/22,202
Embedded 2,900/22,202
Embedded 3,000/22,202
Embedded 3,100/22,202
Embedded 3,200/22,202
Embedded 3,300/22,202
Embedded 3,400/22,202
Embedded 3,500/22,202
Embedded 3,600/22,202
Embedded 3,700/22,202
Embedded 3,800/22,202
Embedded 3,900/22,202
Embedded 4,000/22,202
Embedded 4,100/22,202
Embedded 4,200/22,202
Embedded 4,300/22,202
Embedded 4,400/22,202
Embedded 4,500/22,202
Embedded 4,600/22,202
Embedd

In [5]:
output_table.to_json(
    "./Clean Dataset/pscon_product_table_en.jsonl.gz",
    orient="records",
    lines=True,
    force_ascii=False,
    compression="gzip",
)

## 4. Test-case Construction

In [6]:
product_ids = set(product_table["product_id"])
conversations = pd.read_json("./Raw Dataset/dataset/conversation_en.json")
cases = []

for conversation in conversations.itertuples(index=False):
    history = []

    for message in conversation.conversation:
        if message.get("action") == "Recommend":
            ground_truth = list(dict.fromkeys(
                product["product_id"]
                for product in message.get("recommended_products", [])
            ))

            if ground_truth and all(item in product_ids for item in ground_truth):
                cases.append({
                    "conv_id": conversation.conv_id,
                    "messages": history.copy(),
                    "ground_truth": ground_truth,
                })

        history.append({
            "role": message["role"],
            "content": message["content"].strip(),
        })

test_cases = pd.DataFrame(cases)
test_cases.to_json(
    "./Clean Dataset/pscon_t5_cases_en.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

print(f"Created {len(test_cases):,} cases")
test_cases.head(1)


Created 1,144 cases


,conv_id,messages,ground_truth
0,19762,"[{'role': 'user', 'content': 'Hello, i would l...","[B07JCBG524, B07Y8V4D5X, B01DZQI7B4, B01DZQI6XI]"
